# 01 - Exploratory Data Analysis

This notebook explores the Kaggle Chest X-Ray Pneumonia dataset to understand:
- Dataset structure and class distribution
- Sample images from each class
- Image size distribution
- Class imbalance analysis

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from collections import Counter

DATA_PATH = Path('../data/chest_xray')

## Dataset Structure

In [ ]:
# Count images per split and class
for split in ['train', 'val', 'test']:
    split_path = DATA_PATH / split
    if not split_path.exists():
        print(f'{split}: NOT FOUND')
        continue
    print(f'\n{split.upper()}:')
    for cls in sorted(split_path.iterdir()):
        if cls.is_dir():
            count = len(list(cls.glob('*.jpeg'))) + len(list(cls.glob('*.png'))) + len(list(cls.glob('*.jpg')))
            print(f'  {cls.name}: {count} images')

## Class Distribution

In [ ]:
# Visualize class distribution
train_path = DATA_PATH / 'train'
if train_path.exists():
    normal = len(list((train_path / 'NORMAL').glob('*')))
    pneumonia = len(list((train_path / 'PNEUMONIA').glob('*')))
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # Bar chart
    ax[0].bar(['NORMAL', 'PNEUMONIA'], [normal, pneumonia], color=['#4CAF50', '#f44336'])
    ax[0].set_title('Training Set Class Distribution')
    ax[0].set_ylabel('Number of Images')
    for i, v in enumerate([normal, pneumonia]):
        ax[0].text(i, v + 30, str(v), ha='center', fontweight='bold')
    
    # Pie chart
    ax[1].pie([normal, pneumonia], labels=['NORMAL', 'PNEUMONIA'], 
              autopct='%1.1f%%', colors=['#4CAF50', '#f44336'])
    ax[1].set_title('Class Ratio')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\nImbalance ratio (Pneumonia:Normal): {pneumonia/normal:.2f}:1')
else:
    print('Dataset not found. Download from Kaggle first.')

## Sample Images

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, label in enumerate(['NORMAL', 'PNEUMONIA']):
    folder = train_path / label
    if folder.exists():
        images = sorted(folder.glob('*.jpeg'))[:4]
        for col, img_path in enumerate(images):
            img = Image.open(img_path)
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].set_title(f'{label}\n{img.size[0]}x{img.size[1]}', fontsize=10)
            axes[row, col].axis('off')

plt.suptitle('Sample Chest X-Ray Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## Image Size Distribution

In [ ]:
# Analyze image dimensions
widths, heights = [], []
for img_path in list((train_path / 'NORMAL').glob('*.jpeg'))[:200] + \
                list((train_path / 'PNEUMONIA').glob('*.jpeg'))[:200]:
    img = Image.open(img_path)
    widths.append(img.size[0])
    heights.append(img.size[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', alpha=0.7)
axes[0].set_title('Width Distribution')
axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=30, color='steelblue', alpha=0.7)
axes[1].set_title('Height Distribution')
axes[1].set_xlabel('Pixels')
plt.tight_layout()
plt.show()

print(f'Width  - Mean: {np.mean(widths):.0f}, Min: {np.min(widths)}, Max: {np.max(widths)}')
print(f'Height - Mean: {np.mean(heights):.0f}, Min: {np.min(heights)}, Max: {np.max(heights)}')